In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install -q transformers[sentencepiece]>=4.49.0 datasets>=2.20.0 accelerate>=0.28.0 sentencepiece scikit-learn matplotlib seaborn

In [3]:
import transformers;
print(transformers.__version__)

5.0.0


In [4]:
import os
os.makedirs("/content/drive/MyDrive/255/outputs", exist_ok=True)
os.makedirs("/content/drive/MyDrive/255/plots", exist_ok=True)
os.makedirs("/content/drive/MyDrive/255/model", exist_ok=True)

In [5]:
from transformers import AutoModel
import torch
m = AutoModel.from_pretrained("microsoft/deberta-v3-base", torch_dtype=torch.float32)
print(next(m.parameters()).dtype)  # must print torch.float32


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.float32


In [ ]:
!cp /content/drive/MyDrive/255/l3_deberta_finetune_v2.py /content/

!python /content/drive/MyDrive/255/l3_deberta_finetune_v2.py \
    --data_dir /content/drive/MyDrive/255 \
    --output_root /content/drive/MyDrive/255 \
    --model_name microsoft/deberta-v3-base \
    --fp16 \
    --max_length 256 \
    --batch_size 16 \
    --grad_accum 4 \
    --lr 1e-5 \
    --head_lr 5e-5 \
    --weight_decay 0.05 \
    --epochs 3 \
    --patience 1

Device       : cuda
GPU          : Tesla T4
VRAM         : 15.6 GB
Loading /content/drive/MyDrive/255/reviews_enriched.csv ...
  Dropped 1,272 empty/short rows; 529,859 remain
Loading /content/drive/MyDrive/255/reviewer_profiles.csv ...
Loading /content/drive/MyDrive/255/seller_profiles.csv ...
Total reviews : 529,859
Spam rate     : 13.5%
Meta features : 27
Train : 422,704 (13.5% spam)
Val   : 53,468 (13.2% spam)
Test  : 53,687 (13.3% spam)

Tokenizer: microsoft/deberta-v3-base
tokenizer_config.json: 100% 52.0/52.0 [00:00<00:00, 182kB/s]
spm.model: 100% 2.46M/2.46M [00:00<00:00, 5.83MB/s]
Class weights: legit=0.271  spam=1.729
Focal gamma  : 1.5

Building model: microsoft/deberta-v3-base + 27-dim meta MLP
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 198/198 [00:00<00:00, 1051.95it/s, Materializing param=encoder.rel_embeddings.weight]
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
--------

In [15]:
import os

print("Checking output directory:")
!ls -lt /content/drive/MyDrive/255/outputs

print("\nChecking model directory:")
!ls -lt /content/drive/MyDrive/255/model

Checking output directory:
total 19149
-rw------- 1 root root      136 Apr 26 17:47 deberta_threshold_metadata.json
-rw------- 1 root root      556 Apr 26 17:47 deberta_metrics.json
-rw------- 1 root root 19599751 Apr 26 17:47 deberta_predictions.csv
-rw------- 1 root root     6227 Apr 26 17:09 training_history.csv

Checking model directory:
total 547238
-rw------- 1 root root 556799536 Apr 26 17:47 model.safetensors
-rw------- 1 root root       419 Apr 26 17:47 tokenizer_config.json
-rw------- 1 root root   3559844 Apr 26 17:47 tokenizer.json
-rw------- 1 root root      5201 Apr 26 17:47 training_args.bin
-rw------- 1 root root       848 Apr 26 17:47 config.json
drwx------ 4 root root      4096 Apr 26 17:09 checkpoints


In [16]:
import json
import pandas as pd
import os

metrics_path = '/content/drive/MyDrive/255/outputs/deberta_metrics.json'
history_path = '/content/drive/MyDrive/255/outputs/training_history.csv'

if os.path.exists(metrics_path):
    print("--- Final Training Metrics ---")
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print("Metrics file not found.")

if os.path.exists(history_path):
    print("\n--- Last 5 logs in Training History ---")
    df_history = pd.read_csv(history_path)
    display(df_history.tail())
else:
    print("Training history file not found.")

--- Final Training Metrics ---
{
  "model": "microsoft/deberta-base",
  "unfreeze_layers": 12,
  "trainable_params": 99802370,
  "total_params": 139193858,
  "max_length": 256,
  "epochs_trained": 6,
  "batch_size": 96,
  "grad_accum": 1,
  "effective_batch_size": 96,
  "learning_rate": 4e-05,
  "train_size": 422704,
  "val_size": 53468,
  "test_size": 53687,
  "auc_roc": 0.7382,
  "avg_precision": 0.3152,
  "f1_macro_default": 0.6128,
  "f1_spam_default": 0.3157,
  "optimal_threshold": 0.1634,
  "f1_macro_optimal": 0.3687,
  "f1_spam_optimal": 0.3687,
  "runtime_minutes": 515.9
}

--- Last 5 logs in Training History ---


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_f1_macro,eval_f1_spam,eval_auc_roc,eval_avg_precision,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
54,0.187345,5.845373,4.468243e-07,5.676658,25000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55,0.189095,4.283572,1.886819e-07,5.790191,25500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,0.189480,3.394807,3.988084e-08,5.903724,26000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,NaN,NaN,NaN,6.000000,26424,0.464058,0.607759,0.305626,0.736474,0.306112,203.8476,262.294,1.369,NaN,NaN,NaN,NaN,NaN
58,NaN,NaN,NaN,6.000000,26424,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30956.6446,81.928,0.854,3.887995e+17,0.286698


In [17]:
!cat /content/drive/MyDrive/255/outputs/deberta_metrics.json

{
  "model": "microsoft/deberta-base",
  "unfreeze_layers": 12,
  "trainable_params": 99802370,
  "total_params": 139193858,
  "max_length": 256,
  "epochs_trained": 6,
  "batch_size": 96,
  "grad_accum": 1,
  "effective_batch_size": 96,
  "learning_rate": 4e-05,
  "train_size": 422704,
  "val_size": 53468,
  "test_size": 53687,
  "auc_roc": 0.7382,
  "avg_precision": 0.3152,
  "f1_macro_default": 0.6128,
  "f1_spam_default": 0.3157,
  "optimal_threshold": 0.1634,
  "f1_macro_optimal": 0.3687,
  "f1_spam_optimal": 0.3687,
  "runtime_minutes": 515.9
}

In [18]:
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoConfig

model_path = '/content/drive/MyDrive/255/model'

print(f"Checking model files in {model_path}...")
files = os.listdir(model_path)
for f in ['config.json', 'model.safetensors', 'training_args.bin']:
    if f in files:
        size_mb = os.path.getsize(os.path.join(model_path, f)) / (1024*1024)
        print(f"- {f}: Found ({size_mb:.2f} MB)")
    else:
        print(f"- {f}: MISSING")

try:
    print("\nAttempting to load the saved model to verify integrity...")
    config = AutoConfig.from_pretrained(model_path)
    # Loading with device_map='cpu' to avoid OOM just for a check
    model = AutoModelForSequenceClassification.from_pretrained(model_path, config=config, torch_dtype=torch.float16, low_cpu_mem_usage=True)
    print("✅ Model loaded successfully. The save seems complete.")
except Exception as e:
    print(f"❌ Error loading model: {e}")

Checking model files in /content/drive/MyDrive/255/model...
- config.json: Found (0.00 MB)
- model.safetensors: Found (531.01 MB)
- training_args.bin: Found (0.00 MB)

Attempting to load the saved model to verify integrity...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Model loaded successfully. The save seems complete.


In [19]:
import os
import re

checkpoint_dir = '/content/drive/MyDrive/255/model/checkpoints'

if os.path.exists(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')]

    def get_step(name):
        match = re.search(r'checkpoint-(\d+)', name)
        return int(match.group(1)) if match else 0

    if checkpoints:
        sorted_checkpoints = sorted(checkpoints, key=get_step, reverse=True)
        print(f"Found {len(checkpoints)} checkpoints.")
        print(f"Latest checkpoint (by step number): {sorted_checkpoints[0]}")
        print("\nAll checkpoints found:")
        for ckpt in sorted_checkpoints:
            print(f"- {ckpt}")
    else:
        print("No checkpoint directories found starting with 'checkpoint-'.")
else:
    print(f"Directory not found: {checkpoint_dir}")

Found 2 checkpoints.
Latest checkpoint (by step number): checkpoint-26424

All checkpoints found:
- checkpoint-26424
- checkpoint-22020


In [6]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/255/reviews_enriched.csv",
                 usecols=["review_text"], engine="python", on_bad_lines="skip")
df = df.dropna()
lens = df["review_text"].astype(str).str.split().str.len()
print(lens.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

count    531184.000000
mean        100.625087
std          88.651928
min           1.000000
50%          76.000000
90%         210.000000
95%         271.000000
99%         423.000000
max         983.000000
Name: review_text, dtype: float64


In [8]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("microsoft/deberta-base")
sample = df["review_text"].astype(str).sample(20000, random_state=42).tolist()
tok_lens = [len(tok(t, truncation=False)["input_ids"]) for t in sample]
import numpy as np
for p in [50, 90, 95, 99]:
    print(f"p{p}: {np.percentile(tok_lens, p):.0f}")
print(f"max: {max(tok_lens)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

p50: 100
p90: 274
p95: 352
p99: 548
max: 1273
